# 第22讲：沿用已有结果，验证发布、同步与恢复
此入口调用同目录 run.sh，与 CLI 共用全部逻辑。它不创建新的第21讲候选，也不覆盖来源目录。
direct 使用已经发布的文件；validated 才执行原生验证与发布决定。来源有失败或采集摘要冲突时停止，不用重跑来挑成功。
默认 poll 在没有新模型消息的情况下观察两次原生轮询；每个任务复制自己的 Skill 文件，不在中途替换。

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples/22-skillclaw-shared-revision/run.sh').is_file())
sys.path.insert(0, str(ROOT / 'examples'))
from skillclaw_lab.runtime import run_cli
CLI = ROOT / 'examples/22-skillclaw-shared-revision/run.sh'


In [ ]:
# 与终端 --offline 完全相同；原生同步测试不连接模型。
_ = run_cli(CLI, ['--offline'], timeout=90)


In [ ]:
ALLOW_PAID = False
SOURCE_RUN = None  # 已完成第21讲的真实 run，不是 seed，也不是第22讲恢复后的目录。
SYNC_MODE = 'poll'  # 对照手动拉取可选 manual。
if ALLOW_PAID:
    if SOURCE_RUN is None:
        raise ValueError('请明确指定第21讲来源目录；不会自动重新创建候选')
    _ = run_cli(CLI, ['--source-run', str(Path(SOURCE_RUN).expanduser().resolve()), '--sync-mode', SYNC_MODE])
else:
    print('未授权付费，本单元格不运行验证或业务会话。')


## 从终端给出的 REPORT.md 查看结果
分别看正式文件、接收端文件、skill_view 实际读取、完整查询回执和条件动作。上游异常仍查询连接池就是偏差，不能因 PRM 通过而改判。
每项任务的 input.json 保存实际 system/user 提示。两种现场使用相同的一轮初查要求，按 Skill 正文选择分支，不把全部工具当成必做清单；工具权限没有按场景删减。
恢复是重新 push 来源保存的旧内容、同步并启动独立新会话，不是 restore hermes，也不换成仓库当前 seed。恢复后的查询只记录观察，不宣称行为退回或能力收益。
同一来源再次执行会创建另一个独立副本并再次付费，不是无费用恢复运行。请先读已有报告，不要重复执行在线单元格。